# SLNP — Bayes + microfluidics (short loop)

Same workflow as the lab notebooks: config → connect → valves → calibrate → fill → WORK → schedule → suggest.

Requires: `pip install -e ../..` from the repo root and `smart_pump`.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
ROOT = HERE.parents[1] if HERE.name == "SLNP_bayes" else HERE
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from bayesian_optimization import DesignSpace, ScalarEISuggester
from bayesian_optimization.microfluidics import (
    MixerConfig,
    MixerSession,
    ScheduleRunner,
    connect_mf,
    create_pump_client,
    run_schedule_csv,
)

COM_PORT = "COM9"  # Device Manager
CONFIG = HERE / "sample_config.yml"
DATA = HERE / "data_bayes"
DATA.mkdir(exist_ok=True)
ALL = list(range(8))

## 1. Connect MF

In [ ]:
MF = create_pump_client(CONFIG)
MF.SetSyringeNames({0: "tween", 1: "peg", 2: "lipid", 3: "ley", 4: "ctab", 5: "pvp", 6: "cit", 7: "sep"})
await connect_mf(MF, COM_PORT)
MF.messenger.debug = False

## 2. Valves FILL (1) → calibrate → fill → WORK (0)

`0` = system, `1` = syringe fill

In [ ]:
for i in ALL:
    await MF.ValveChangeOne(i, 1)

await MF.Calibrate()
print("positions:", await MF.GetPositionsMF())

In [ ]:
# fill (run Teos/wash as a separate bench step if needed)
await MF.Stop()
await MF.SetSpeedAndVolumeToMove(ALL, [150] * 8, [9400] * 8)
await MF.Start()
# wait until filled, then Stop + WORK valves

In [ ]:
await MF.Stop()
for i in ALL:
    await MF.ValveChangeOne(i, 0)  # WORK

## 3. Bayes: LHS → CSV

In [ ]:
cfg = MixerConfig(
    data_dir=str(DATA),
    var_to_syringe=dict(TWEEN=0, PEG=1, LIPID=2, LEY=3),
    bounds={k: (1.0, 40.0) for k in ["TWEEN", "PEG", "LIPID", "LEY"]},
    n_syringes=8,
    total_speed=50.0,
    time_synth=30.0,
    default_n_points=10,
    random_state=0,
    separator_syringe=7,
    separator_speed=0.0,
    time_separator=20.0,
    legacy_csv=True,
)
space = DesignSpace(names=list(cfg.var_to_syringe), bounds=cfg.bounds, sum_equals=cfg.total_speed)
mixer = MixerSession(cfg, ScalarEISuggester(space, maximize=True))
recipe = mixer.generate_lhs_iter0(n_points=10)
print(recipe)


## 4. Run schedule (`ScheduleRunner` unchanged)

In [ ]:
runner = ScheduleRunner(MF, printer=None, pumps=tuple(ALL), tick_sleep=0.1)
await run_schedule_csv(runner, recipe, n_syringes=8)

## 5. After scoring → next iteration

Write `recipes_iter_000_results.csv` (score in the last column on recipe rows), then:

In [ ]:
# next_csv = mixer.suggest_next(iter_idx=0, n_points=10)
# print(next_csv)
# await run_schedule_csv(runner, next_csv, n_syringes=8)